# Notebook 5: Experiment 1 — Same-Stock Prediction (70/30)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on each stock's daily data and predict its own future prices.  
**Train/Test Split:** 70/30 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Stocks:** TLKM, BBCA, ASII, UNVR  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.7
RATIO_LABEL = '70_30'
EXP_LABEL = f'Exp1_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 1 - Same Stock Prediction (70/30)")
print(f"Train ratio: {TRAIN_RATIO}, Test ratio: {1-TRAIN_RATIO}")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 60, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 1 - Same Stock Prediction (70/30)
Train ratio: 0.7, Test ratio: 0.30000000000000004


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Run All Experiments

In [3]:
# ============================================================
# EXPERIMENT 1: Train and predict on same stock
# ============================================================
all_results = []
all_predictions = {}  # {stock: {model_type: (y_true, y_pred, dates)}}
all_histories = {}    # {stock: {model_type: history}}

for stock in STOCKS:
    print(f"\n############################################################")
    print(f"# STOCK: {stock}")
    print(f"############################################################")
    
    # Prepare data
    X_train, y_train, X_test, y_test, test_dates = prepare_same_stock_data(
        daily_data[stock], train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[stock] = {}
    all_histories[stock] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_{stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        # Store results
        result = {'Stock': stock, 'Model': model_type, **metrics}
        all_results.append(result)
        all_predictions[stock][model_type] = (y_true_inv, y_pred_inv, test_dates)
        all_histories[stock][model_type] = history
        
        # Plot individual prediction
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )
        
        # Plot training history
        plot_training_history(
            history, model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 1 (70/30) training complete!")



############################################################
# STOCK: TLKM
############################################################
  X_train: (3610, 60, 1), X_test: (1573, 60, 1)

Training BiLSTM for: Exp1_70_30_TLKM
  Train samples: 3610, Test samples: 1573
Epoch 1/100
51/51 [==============================] - ETA: 0s - loss: 0.0012
Epoch 1: val_loss improved from inf to 0.00017, saving model to models/Exp1_70_30\Exp1_70_30_TLKM_BiLSTM_best.keras
51/51 [==============================] - 15s 63ms/step - loss: 0.0012 - val_loss: 1.7056e-04
Epoch 2/100
51/51 [==============================] - ETA: 0s - loss: 1.2990e-04
Epoch 2: val_loss improved from 0.00017 to 0.00015, saving model to models/Exp1_70_30\Exp1_70_30_TLKM_BiLSTM_best.keras
51/51 [==============================] - 2s 34ms/step - loss: 1.2990e-04 - val_loss: 1.5144e-04
Epoch 3/100
51/51 [==============================] - ETA: 0s - loss: 1.1113e-04
Epoch 3: val_loss improved from 0.00015 to 0.00014, saving model to models

## Results Summary

In [4]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 1 - Same Stock Prediction (70/30)")

# Save results
results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 1 - Same Stock Prediction (70/30)
Stock  Model         MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
 TLKM BiLSTM  14309.8716 119.6239 100.0160    3.2762 0.932060            174.0         100
 TLKM  BiGRU   5654.3108  75.1952  57.5024    1.9577 0.973154            164.7         100
 TLKM   LSTM   9025.2442  95.0013  74.9748    2.5497 0.957150             98.3         100
 TLKM    GRU   6508.4116  80.6747  63.5888    2.1416 0.969099             91.7         100
 BBCA BiLSTM  88968.4686 298.2758 233.9387    3.0760 0.964381            167.5         100
 BBCA  BiGRU 151495.2837 389.2240 324.9464    4.2954 0.939349            164.1         100
 BBCA   LSTM  60468.1863 245.9028 201.4753    2.7978 0.975792             95.2         100
 BBCA    GRU  80726.8969 284.1248 241.1795    3.2948 0.967681             90.9         100
 ASII BiLSTM  12463.0261 111.6379  82.6405    1.9681 0.976578            169.8         100
 ASII  BiGRU  11623.6506 107.8130  81.3610

## Visualizations

In [5]:
# ============================================================
# ALL MODELS COMPARISON PER STOCK
# ============================================================
for stock in STOCKS:
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    preds = {mt: all_predictions[stock][mt][1] for mt in MODEL_TYPES}
    
    plot_all_models_comparison(
        dates, y_true, preds, stock, EXP_LABEL,
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All comparison plots saved!")


  Figure saved: figures/Exp1_70_30/Exp1_70_30_TLKM_all_models.png
  Figure saved: figures/Exp1_70_30/Exp1_70_30_BBCA_all_models.png
  Figure saved: figures/Exp1_70_30/Exp1_70_30_ASII_all_models.png
  Figure saved: figures/Exp1_70_30/Exp1_70_30_UNVR_all_models.png
All comparison plots saved!


In [6]:
# ============================================================
# METRICS BAR CHARTS
# ============================================================
for metric in ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_comparison_bar(
        results_df, metric, EXP_LABEL,
        group_col='Stock', save_dir=f'figures/{EXP_LABEL}'
    )

print("All metrics bar charts saved!")


  Figure saved: figures/Exp1_70_30/Exp1_70_30_MSE_comparison.png
  Figure saved: figures/Exp1_70_30/Exp1_70_30_RMSE_comparison.png
  Figure saved: figures/Exp1_70_30/Exp1_70_30_MAE_comparison.png
  Figure saved: figures/Exp1_70_30/Exp1_70_30_MAPE_pct_comparison.png
  Figure saved: figures/Exp1_70_30/Exp1_70_30_R2_comparison.png
All metrics bar charts saved!


In [7]:
# ============================================================
# SUMMARY: BEST MODEL PER STOCK
# ============================================================
print("\n" + "="*60)
print("  BEST MODEL PER STOCK (by RMSE)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['RMSE'].idxmin()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")

print("\n  BEST MODEL PER STOCK (by R² Score)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['R2'].idxmax()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (R²={best['R2']:.6f}, RMSE={best['RMSE']:.4f})")



  BEST MODEL PER STOCK (by RMSE)
  TLKM: BiGRU (RMSE=75.1952, R²=0.973154)
  BBCA: LSTM (RMSE=245.9028, R²=0.975792)
  ASII: GRU (RMSE=95.6323, R²=0.982813)
  UNVR: GRU (RMSE=116.4155, R²=0.995446)

  BEST MODEL PER STOCK (by R² Score)
  TLKM: BiGRU (R²=0.973154, RMSE=75.1952)
  BBCA: LSTM (R²=0.975792, RMSE=245.9028)
  ASII: GRU (R²=0.982813, RMSE=95.6323)
  UNVR: GRU (R²=0.995446, RMSE=116.4155)
